In [1]:
# ===== CELL 0: تثبيت المكتبات =====
# ⚠️ تأكد إنك شغّال على T4 GPU:
# Runtime > Change runtime type > T4 GPU > Save
# بعدين شغّل الـ cell دي لوحدها وبعد ما تخلص:
# Runtime > Restart session
# وبعد الـ restart شغّل باقي الـ cells

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    lines = result.stdout.strip().split('\n')
    gpu_line = next((l for l in lines if 'MiB' in l or 'Tesla' in l or 'T4' in l), lines[-1])
    print('✅ GPU detected:', gpu_line.strip())
else:
    print('❌ No GPU! Go to Runtime > Change runtime type > T4 GPU')
    raise SystemExit('Please enable GPU first')

!pip install -q -U transformers datasets peft accelerate
!pip install -q -U 'trl>=0.8.0'
!pip install -q -U 'bitsandbytes>=0.46.1'
!pip install -q scikit-learn tqdm pandas

print()
print('✅ Installation complete!')
print('⚠️  NOW: Runtime > Restart session, then run from Cell 1')


✅ GPU detected: |   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00

✅ Installation complete!
⚠️  NOW: Runtime > Restart session, then run from Cell 1


In [2]:
import os
from google.colab import files

TARGET = 'Entity Recognition in Resumes.json'

if not os.path.exists(TARGET):
    print('📂 Please upload Entity Recognition in Resumes.json below:')
    uploaded = files.upload()
    # لو اترفع بأي اسم تاني، نعمله rename
    if TARGET not in uploaded:
        for fname in uploaded:
            if fname.endswith('.json'):
                os.rename(fname, TARGET)
                print(f'✅ Renamed: {fname} → {TARGET}')
                break
else:
    print('✅ File already exists')

size = os.path.getsize(TARGET) / 1024
print(f'📄 File size: {size:.1f} KB')

import torch
if torch.cuda.is_available():
    print(f'🖥️  GPU: {torch.cuda.get_device_name(0)}')
else:
    print('❌ GPU NOT FOUND — Go to Runtime > Change runtime type > T4 GPU')


📂 Please upload Entity Recognition in Resumes.json below:


Saving Entity Recognition in Resumes.json to Entity Recognition in Resumes.json
📄 File size: 1192.2 KB
🖥️  GPU: Tesla T4


In [3]:
import json
import pandas as pd
from datasets import Dataset

TARGET = 'Entity Recognition in Resumes.json'
raw_data = []
with open(TARGET, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                raw_data.append(json.loads(line))
            except json.JSONDecodeError:
                continue

print(f'✅ Total resumes loaded: {len(raw_data)}')

FIELDS = ['Name', 'Email Address', 'Skills', 'Education']

def build_annotation_json(annotations):
    result = {f: '' for f in FIELDS}
    for ann in annotations:
        label_raw = ann.get('label', [])
        if not label_raw:
            continue
        label = label_raw[0] if isinstance(label_raw, list) else label_raw
        if not label or label not in result:
            continue
        points = ann.get('points', [])
        if points:
            result[label] = points[0].get('text', '').strip()
    return json.dumps(result, ensure_ascii=False)

def prepare_sample(item):
    content = item['content'].replace('\n', ' ').strip()
    annotation_str = build_annotation_json(item.get('annotation', []))
    prompt = (
        '### Instruction: Extract Name, Email Address, Skills, and Education from the resume.\n'
        f'### Input: {content}\n'
        f'### Response: {annotation_str}'
    )
    return {'text': prompt, 'content': content, 'annotation': annotation_str}

processed = [prepare_sample(item) for item in raw_data]
df = pd.DataFrame(processed)
dataset = Dataset.from_pandas(df)

print(f'📊 Dataset size: {len(dataset)}')
print('\nSample (truncated):')
print(dataset[0]['text'][:400])


✅ Total resumes loaded: 220
📊 Dataset size: 220

Sample (truncated):
### Instruction: Extract Name, Email Address, Skills, and Education from the resume.
### Input: Abhishek Jha Application Development Associate - Accenture  Bengaluru, Karnataka - Email me on Indeed: indeed.com/r/Abhishek-Jha/10e7a8cb732bc43a  • To work for an organization which provides me the opportunity to improve my skills and knowledge for my individual and company's growth in best possible wa


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

assert torch.cuda.is_available(), '❌ GPU not found! Go to Runtime > Change runtime type > T4 GPU'
print(f'✅ Using GPU: {torch.cuda.get_device_name(0)}')
print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float32,  # float32 بدل float16 لتفادي الـ BFloat16 conflict
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

print('✅ Model loaded on', model.device)


✅ Using GPU: Tesla T4
   VRAM: 15.6 GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded on cuda:0


In [5]:
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

sft_config = SFTConfig(
    output_dir='./results',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    save_strategy='epoch',
    logging_steps=10,
    fp16=False,               # ❌ أوقفنا fp16
    bf16=False,               # ❌ أوقفنا bf16 (T4 مش بيدعمها كويس مع 4-bit)
    dataset_text_field='text',
    max_length=512,           # قللنا من 1024 لـ 512 لتفادي الـ OOM على T4
    report_to='none',
    optim='paged_adamw_8bit', # optimizer أكثر كفاءة في الذاكرة
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=sft_config,
)

print('🚀 Starting training...')
trainer.train()

trainer.model.save_pretrained('./final-resume-model')
tokenizer.save_pretrained('./final-resume-model')
print('✅ Model saved!')


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/220 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/220 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2152 > 2048). Running this sequence through the model will result in indexing errors
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


🚀 Starting training...


Step,Training Loss
10,2.434960
20,2.232518
30,2.165276
40,2.034983


✅ Model saved!


In [6]:
import torch
from tqdm import tqdm
from sklearn.metrics import classification_report

def evaluate_model_flexible(test_data, model, tokenizer, num_samples=20):
    all_preds_binary = []
    all_actuals_binary = []
    label_map = {
        'Name': ['name', 'candidate name', 'full name'],
        'Email Address': ['email', 'email address', 'contact'],
        'Skills': ['skills', 'technical skills', 'skill'],
        'Education': ['education', 'degree', 'qualification'],
    }
    print('Starting Evaluation... 🔍')
    model.eval()
    for item in tqdm(test_data[:num_samples]):
        content = item['content'].replace('\n', ' ')
        prompt = (
            '### Instruction: Extract Name, Email Address, Skills, and Education from the resume.\n\n'
            f'### Input: {content}\n\n'
            '### Response:'
        )
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=False)
        full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response_text = (
            full_output.split('### Response:')[1].lower()
            if '### Response:' in full_output else full_output.lower()
        )
        actual_values = {}
        for ann in item.get('annotation', []):
            label_raw = ann.get('label', [])
            if not label_raw:
                continue
            label = label_raw[0] if isinstance(label_raw, list) else label_raw
            if label in label_map and ann.get('points'):
                actual_values[label] = ann['points'][0]['text'].strip().lower()
        for field in label_map:
            if field in actual_values:
                actual_val = actual_values[field]
                is_found = 1 if (actual_val and actual_val in response_text) else 0
                all_preds_binary.append(is_found)
                all_actuals_binary.append(1)
    print('\n' + '='*45)
    print('📊 F1-SCORE REPORT (Partial Match)')
    print('='*45)
    print(classification_report(
        all_actuals_binary, all_preds_binary,
        target_names=['Not Found', 'Found'],
        zero_division=0
    ))

evaluate_model_flexible(raw_data, model, tokenizer)


Starting Evaluation... 🔍


  0%|          | 0/20 [00:00<?, ?it/s][transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 20/20 [04:34<00:00, 13.71s/it]


📊 F1-SCORE REPORT (Partial Match)
              precision    recall  f1-score   support

   Not Found       0.00      0.00      0.00         0
       Found       1.00      0.59      0.74        54

    accuracy                           0.59        54
   macro avg       0.50      0.30      0.37        54
weighted avg       1.00      0.59      0.74        54



In [7]:
from google.colab import files

# ضغط وتحميل الموديل
!zip -r model.zip ./final-resume-model
files.download("model.zip")
print("Download started ✅")

  adding: final-resume-model/ (stored 0%)
  adding: final-resume-model/README.md (deflated 65%)
  adding: final-resume-model/tokenizer.json (deflated 85%)
  adding: final-resume-model/chat_template.jinja (deflated 60%)
  adding: final-resume-model/tokenizer_config.json (deflated 46%)
  adding: final-resume-model/adapter_model.safetensors (deflated 22%)
  adding: final-resume-model/adapter_config.json (deflated 58%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started ✅


In [10]:
!pip install -q -U torchao
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 1. تحميل الموديل الأصلي
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# 2. تحميل الـ LoRA adapters
model = PeftModel.from_pretrained(base_model, "./final-resume-model")
model.eval()
print("Fine-tuned model loaded ✅")

# 3. تجربة استخراج بيانات من سيرة ذاتية
resume_text = "John Doe, email: john@example.com. Skills: Python, SQL, Machine Learning. Education: BS in Computer Science from Cairo University."
prompt = (
    f"### Instruction: Extract Name, Email Address, Skills, and Education from the resume.\n\n"
    f"### Input: {resume_text}\n\n"
    f"### Response:"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n--- Output ---")
print(result.split("### Response:")[1].strip() if "### Response:" in result else result)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.6 MB/s eta 0:00:00


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Fine-tuned model loaded ✅

--- Output ---
Name: John Doe
Email Address: john@example.com
Skills: Python, SQL, Machine Learning
Education: BS in Computer Science from Cairo University
